# 🚀 UniqToken: Interactive Quickstart & Multilingual Tokenization Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umran666/UniqToken/blob/main/notebooks/quickstart.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-UniqToken-181717.svg?logo=github)](https://github.com/umran666/UniqToken)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)

Welcome to the **UniqToken** interactive Google Colab tutorial!

**UniqToken** is a high-precision, script-aware, entropy-guided multilingual subword tokenizer designed to eliminate the **"Token Tax"** in modern Large Language Models.

### Key Capabilities
- 🌐 **Script-Aware Tokenization**: Native protection for Indic viramas, Arabic harakat, Hebrew niqqud, and Hangul jamo clusters.
- ⚡ **Byte-Fallback Engine**: 0% Out-Of-Vocabulary (OOV) across all Unicode code points and arbitrary byte sequences.
- 🎯 **Exact Dual-Offset Span Tracking**: Tracks character spans from raw text → normalized text → token spans for precision entity alignment and NER.
- 🏎️ **Native Rust Acceleration**: Optional Rust acceleration core (`crates/uniqtoken_core`) providing multi-threaded Rayon processing and fused Viterbi DAG search.
- 🔄 **Hugging Face & GGUF Interoperability**: Direct export to Hugging Face standard schema `tokenizer.json` and LLaMA.cpp GGUF v3 format.

---

## 📦 Step 1: Environment Setup & Installation

Let's clone the repository and install UniqToken along with evaluation dependencies (`tiktoken`, `matplotlib`, `tokenizers`).

In [ ]:
# Clone repository if running in Google Colab
import os
import sys

if not os.path.exists("UniqToken") and not os.path.exists("uniqtoken"):
    !git clone https://github.com/umran666/UniqToken.git
    %cd UniqToken
elif os.path.exists("UniqToken"):
    %cd UniqToken

# Install UniqToken in editable mode along with dependencies
!pip install -q -e . tiktoken matplotlib tokenizers pandas

### 🦀 (Optional) Build Native Rust Acceleration Core

UniqToken runs seamlessly in **pure Python** out of the box with zero compilation required.
For 10x–50x higher throughput on large datasets, you can optionally build the native Rust acceleration core with `maturin`:

In [ ]:
# Compile native Rust acceleration core (optional)
!pip install -q maturin
!maturin develop --release --manifest-path crates/uniqtoken_core/Cargo.toml

# Verify acceleration engine status
try:
    import uniqtoken_core

    print("🚀 Native Rust acceleration core is ACTIVE! (crates/uniqtoken_core compiled)")
except ImportError:
    print("⚡ Running in Pure-Python mode (full parity guaranteed across all algorithms).")

## 🧠 Step 2: Training a Custom Unigram Tokenizer on Multilingual Data

Traditional BPE tokenizers (like GPT-4's Tiktoken or LLaMA BPE) fragment non-Latin scripts (Devanagari, Arabic, CJK) into individual raw UTF-8 byte tokens. This dramatically inflates token counts and increases inference latency and API cost.

UniqToken trains an optimal **Unigram Language Model** (Kudo, 2018) with:
1. **Script-aware pre-tokenization** (isolating scripts, numbers, and whitespaces)
2. **Subword seed vocabulary mining** with character-savings ranking
3. **Expectation-Maximization (EM) pruning** down to your target vocabulary budget
4. **Built-in Byte-Fallback** ensuring 100% token coverage for unseen inputs

In [ ]:
# Diverse multilingual & code sample corpus
sample_corpus = [
    # English & Technical Prose
    "UniqToken is a script-aware, entropy-guided multilingual subword tokenizer.",
    "Tokenization is the foundational first layer of modern Large Language Models.",
    "Attention mechanisms allow transformers to process input tokens in parallel.",
    "Self-attention calculates pairwise relationships between all positions in a sequence.",
    # Python Code with Indentation & Syntax
    "def calculate_fibonacci(n: int) -> int:\n    if n <= 1:\n        return n\n    return calculate_fibonacci(n - 1) + calculate_fibonacci(n - 2)",
    "class DistributedOptimizer:\n    def __init__(self, lr: float = 1e-4):\n        self.lr = lr\n    def step(self):\n        pass",
    "import torch\nimport torch.nn as nn\nmodel = nn.Transformer(d_model=512, nhead=8)",
    # Indic / Hindi (Devanagari Script with Virama Clusters)
    "आर्टिफिशियल इंटेलिजेंस और मशीन लर्निंग आधुनिक तकनीक की सबसे महत्वपूर्ण नींव हैं।",
    "प्राकृतिक भाषा प्रसंस्करण में टोकनाइज़र बहुत महत्वपूर्ण भूमिका निभाता है।",
    "कंप्यूटर विज्ञान और कृत्रिम बुद्धिमत्ता में नए शोध हो रहे हैं।",
    # CJK / Japanese (Kanji, Hiragana, Katakana)
    "自然言語処理におけるトークナイザーの最適化は非常に重要です。",
    "大規模言語モデルは多言語テキストを効率的かつ正確に処理する必要があります。",
    "深層学習アーキテクチャの進化により多言語対応が加速しています。",
    # Arabic (Right-to-Left with Diacritics)
    "تعتبر معالجة اللغات الطبيعية من أهم مجالات الذكاء الاصطناعي الحديثة.",
    "يساعد نموذج يونिक توكن في تقليل عدد الرموز للنصوص العربية بكفاءة عالية.",
    "الخوارزميات الذكية تمكن الحواسيب من فهم اللغات البشرية بدقة.",
    # Agglutinative Morphology (Turkish & Finnish)
    "Muvaffakiyetsizleştiricileştiriveremeyebileceklerimizdenmişsinizcesine.",
    "Afyonkarahisarlılaştırabildiklerimizdenmişsinizcesine bir örnek cümle.",
    "epäjärjestelmällistyttämättömyydelläänsäkäänköhän suomen kielen sana.",
    "Lentokonesuihkuturbiinimoottoriapumekaanikkoaliupseerioppilas.",
]

print(f"Sample corpus size: {len(sample_corpus)} documents")

In [ ]:
from uniqtoken import CustomTokenizer

print("Training UniqToken Unigram Tokenizer...")
tokenizer = CustomTokenizer.train_from_corpus(
    corpus=sample_corpus,
    target_vocab_size=500,
    special_tokens=["<|pad|>", "<|unk|>", "<|bos|>", "<|eos|>"],
    byte_fallback=True,
    verbose=True,
)

print(f"\n✅ Training complete! Vocabulary size: {tokenizer.vocab_size}")

### 🔍 Inspecting Encoding, Decoding & Dual-Offset Tracking

UniqToken tracks exact character-span offsets from original raw text, through Unicode normalization, to token spans.
This guarantees lossless roundtrips and accurate Named Entity Recognition (NER) alignment.

In [ ]:
test_text = "आर्टिफिशियल इंटेलिजेंस def step(self): pass"

# 1. Encode to tokens
tokens = tokenizer.encode(test_text)
print("Tokens:", tokens)

# 2. Encode to integer IDs
token_ids = tokenizer.encode_to_ids(test_text)
print("Token IDs:", token_ids)

# 3. Lossless decode back to string
decoded = tokenizer.decode(token_ids)
print("Decoded Text:", decoded)
assert decoded == test_text, "Roundtrip must match perfectly!"

# 4. Dual-offset character spans
print("\nCharacter Spans:")
for token in tokenizer.encode_with_offsets(test_text)[:8]:
    print(f"  Token: {token.text!r:<20} ID: {token.id:<5} Span: {token.raw_span}")

## 📊 Step 3: Visualizing Token Compression (Bytes / Token) vs OpenAI Tiktoken

### The "Token Tax" Explained
LLMs charge by the token and operate within finite context windows.
- **Bytes per Token (BPT)** measures information density: $\text{BPT} = \frac{\text{UTF-8 Bytes}}{\text{Token Count}}$.
- **Higher BPT is better**: More information is packed into fewer tokens, resulting in lower API inference costs and larger effective context capacity.
- Non-English languages typically suffer low BPT under standard tokenizers due to excessive fragmentation.

Let's run a side-by-side benchmark comparing **UniqToken** vs OpenAI's standard production tokenizer **Tiktoken (`cl100k_base`)**.

In [ ]:
import tiktoken
import pandas as pd
import matplotlib.pyplot as plt

# Load standard OpenAI production tokenizer
tik_enc = tiktoken.get_encoding("cl100k_base")

benchmark_samples = [
    ("English Prose", "Tokenization is the foundational first layer of modern Large Language Models."),
    ("Python Code", "    def calculate_fibonacci(n: int) -> int:\n        return n if n <= 1 else fib(n-1) + fib(n-2)"),
    ("Hindi (Devanagari)", "आर्टिफिशियल इंटेलिजेंस और मशीन लर्निंग आधुनिक तकनीक की नींव हैं।"),
    ("Japanese (CJK)", "自然言語処理におけるトークナイザーの最適化は非常に重要です。"),
    ("Arabic (RTL)", "تعتبر معالجة اللغات الطبيعية من أهم مجالات الذكاء الاصطناعي الحديثة."),
    ("Finnish (Agglutinative)", "epäjärjestelmällistyttämättömyydelläänsäkäänköhän suomen kielen sana."),
]

records = []
for domain, text in benchmark_samples:
    raw_bytes = len(text.encode("utf-8"))

    # UniqToken
    u_tokens = tokenizer.encode(text)
    u_count = len(u_tokens)
    u_bpt = raw_bytes / u_count

    # Tiktoken (cl100k_base)
    t_tokens = tik_enc.encode(text)
    t_count = len(t_tokens)
    t_bpt = raw_bytes / t_count

    # Context Savings (% fewer tokens needed with UniqToken)
    savings = ((t_count - u_count) / t_count) * 100 if t_count else 0.0

    records.append(
        {
            "Domain": domain,
            "Raw Bytes": raw_bytes,
            "UniqToken Count": u_count,
            "UniqToken BPT": round(u_bpt, 2),
            "Tiktoken Count": t_count,
            "Tiktoken BPT": round(t_bpt, 2),
            "Context Savings (%)": f"{savings:+.1f}%",
        }
    )

df = pd.DataFrame(records)
display(df)

In [ ]:
# Visualize Bytes/Token comparison across linguistic domains
plt.figure(figsize=(10, 5), dpi=120)

domains = [r["Domain"] for r in records]
x = range(len(domains))
width = 0.35

u_bpts = [r["UniqToken BPT"] for r in records]
t_bpts = [r["Tiktoken BPT"] for r in records]

plt.bar([i - width / 2 for i in x], u_bpts, width=width, label="UniqToken (Custom Vocab)", color="#2563EB", alpha=0.9)
plt.bar(
    [i + width / 2 for i in x], t_bpts, width=width, label="OpenAI Tiktoken (cl100k_base)", color="#94A3B8", alpha=0.9
)

plt.xlabel("Linguistic Domain / Script Family", fontweight="bold", fontsize=11)
plt.ylabel("Bytes per Token (Higher is Better ↑)", fontweight="bold", fontsize=11)
plt.title("Token Compression Efficiency: UniqToken vs Tiktoken", fontsize=13, fontweight="bold", pad=12)
plt.xticks(x, domains, rotation=15, ha="right", fontsize=9)
plt.legend(frameon=True, facecolor="white", loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 💾 Step 4: Exporting to Hugging Face `tokenizer.json`

UniqToken features native serialization to the canonical Hugging Face `tokenizers` v1.0 schema.
This creates:
- `tokenizer.json`: The complete Unigram model, normalizer pipeline, byte fallback decoders, and special tokens.
- `tokenizer_config.json`: Configuration declaring `UniqTokenizerFast` with auto-mapping for `transformers.AutoTokenizer`.

Let's export and verify direct loading with the official Hugging Face `tokenizers` library!

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

export_path = Path("./hf_export")
tokenizer.export_to_huggingface(export_path)

print(f"Exported files to {export_path.resolve()}:")
for f in export_path.iterdir():
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

# Load directly using the official Hugging Face tokenizers library
hf_tok = Tokenizer.from_file(str(export_path / "tokenizer.json"))
print("\n✅ Successfully loaded tokenizer.json with Hugging Face Tokenizer!")
print(f"Hugging Face vocabulary size: {hf_tok.get_vocab_size()}")

# Test encoding and validate parity against UniqToken
test_sample = "UniqToken multilingual tokenization"
encoded_hf = hf_tok.encode(test_sample)
u_ids = tokenizer.encode_to_ids(test_sample)
print("HF Token IDs:       ", encoded_hf.ids)
print("UniqToken Token IDs:", u_ids)
assert encoded_hf.ids == u_ids, "Exported Hugging Face tokenizer IDs must match UniqToken native encode!"
print("\n✅ Parity verified: Hugging Face Tokenizer outputs match UniqToken native encode!")

## 🚀 Step 5: Advanced Features & Edge Deployment

### 1. Cross-Entropy Merging (SuperBPE)
UniqToken supports online vocabulary expansion and SuperBPE whitespace-crossing merges (Liu et al., 2025):

In [ ]:
from uniqtoken import CrossEntropyMerging

# Expand vocabulary by mining cross-word merges that reduce cross-entropy
cem = CrossEntropyMerging(max_merges=50, cross_word=True, verbose=False)
superbpe_model = cem.optimize(tokenizer.model, chunks=sample_corpus)

superbpe_tok = CustomTokenizer(
    normalizer=tokenizer.normalizer,
    pre_tokenizer=tokenizer.pre_tokenizer,
    model=superbpe_model,
)

print(f"SuperBPE vocabulary expanded from {tokenizer.vocab_size} to {superbpe_tok.vocab_size} subwords!")

### 2. GGUF Binary Export for `llama.cpp`
Export your trained tokenizer directly into a binary GGUF v3 file for edge inference in `llama.cpp` or Ollama:

In [ ]:
gguf_path = "model_vocab.gguf"
superbpe_tok.export_to_gguf(gguf_path, model_name="uniqtoken_superbpe")
print(f"✅ Exported SuperBPE GGUF v3 binary table to {gguf_path} ({os.path.getsize(gguf_path) / 1024:.1f} KB)")

---
## 🎉 Congratulations!

You have completed the UniqToken Google Colab tutorial!

### What You Learned:
1. Setting up UniqToken with optional native Rust acceleration.
2. Training a script-aware Unigram tokenizer on multilingual & code data.
3. Quantifying token compression (Bytes/Token) against standard OpenAI Tiktoken.
4. Exporting models directly to standard Hugging Face `tokenizer.json` and LLaMA.cpp GGUF formats.

### Useful Links:
- 📖 [UniqToken GitHub Repository](https://github.com/umran666/UniqToken)
- 📝 [Scientific Paper Draft](https://github.com/umran666/UniqToken/blob/main/PAPER_DRAFT.md)
- 🐛 [Report an Issue](https://github.com/umran666/UniqToken/issues)
